# Federated Statistics for Financial Fraud Detection
This notebook demonstrates how to compute federated statistics on financial transaction data using NVIDIA FLARE (NVFlare). It shows how to define statistical recipes, execute them in a federated environment, and visualize the results from different distributed sites.

## 0. Define User Info
In production, provide your local NVFlare startup kit and username information.

In [ ]:
#startup_kit_location = "/path/to/admin_startup_kit"
#username = "admin@nvidia.com"

## 1. Define Stats Recipe

In [ ]:
from stats.client import FinancialStatistics
from misc.data import all_model_parameters, flag, all_data_features

from nvflare.recipe.fedstats import FedStatsRecipe
from nvflare.recipe import SimEnv, ProdEnv

# Local simulation path
site_names = ["siteA", "siteB", "siteC", "siteD", "siteE"]

data_selection="sim-exp"
# data paths are defined in misc/experiments.py

df_stats_generator = FinancialStatistics(
    data_selection=data_selection,
    data_features=all_model_parameters + [flag],
) 
# client name will be used as prefix for the data paths

statistic_configs = {
    "count": {},
    "mean": {},
    "sum": {},
    "stddev": {},
    "histogram": {"*": {"bins": 10, "range": [-1, 1]}, flag: {"bins": 2, "range": [0, 1]}}, 
    "quantile": {"*": [0.1, 0.5, 0.9]},
}

# Define recipe
recipe = FedStatsRecipe(
    name="fedstats",
    sites=site_names,
    statistic_configs=statistic_configs,
    stats_generator=df_stats_generator,
    stats_output_path="statistics/financial_stats.json"
)

# optionally export
recipe.export("job_configs")

## 2. Run in Simulation

In [ ]:
env = SimEnv(clients=site_names, num_threads=len(site_names))
run = recipe.execute(env=env)

## 3. (Optionally) Run in Production Environment

In [ ]:
#env = ProdEnv(startup_kit_location=startup_kit_location, username=username)
#run = recipe.execute(env=env)

### Get Status

In [ ]:
print("Job Status is:", run.get_status())

### Get Results

In [ ]:
result_path = run.get_result()
print("Result can be found in:", result_path)

## 3. Show stats

In [ ]:
stats_file = result_path + '/server/simulate_job/statistics/financial_stats.json'
!head -c 200 {stats_file}

In [ ]:
import json

from nvflare.app_opt.statistics.visualization.statistics_visualization import Visualization

with open(stats_file, 'r') as f:
    data = json.load(f)

### Overall stats

In [ ]:
vis = Visualization()
vis.show_stats(data = data)

## Histogram Visualization

(see [here](https://github.com/NVIDIA/NVFlare/blob/main/examples/advanced/federated-statistics/df_stats/demo/visualization.ipynb) for more display options)

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100%  depth:100% !important; }</style>"))

vis.show_histograms(data = data) # display_format = "percent"